# 12 — 推荐图传播诊断：信号、噪声与长尾

这一课不急着换更复杂的模型，而是回答：**GNN 在什么推荐图上有用，又会在什么情况下扩散噪声？** 我们将推荐目标固定为评分 ≥4，只改变构图阈值、传播深度和层权重，并报告多随机种子、覆盖率及头部/长尾 Recall。

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from crosscity.data.recommendation import load_movielens_implicit
from crosscity.models.recommendation import build_lightgcn
from crosscity.training.recommendation import ranking_diagnostics, train_recommender
from crosscity.utils import seed_everything
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 1. 先分开两个阈值

`positive_rating=4` 定义什么是需要预测的明确偏好；`graph_rating` 定义哪些历史行为可以作为消息传递边。`graph_rating=3` 不是把 3 分当成推荐成功，而是假设中性评分也可能说明用户接触过某类电影。验证目标边会从传播图移除，测试边原本就不在训练图中。

In [ ]:
datasets = {threshold: load_movielens_implicit(
    ROOT / 'data/raw/movielens100k', positive_rating=4, graph_rating=threshold
) for threshold in (4, 3)}
graph_rows = []
for threshold, data in datasets.items():
    undirected_edges = data.edge_index.size(1) // 2
    item_nodes = data.edge_index[1, data.edge_index[1] >= data.num_users] - data.num_users
    degree = torch.bincount(item_nodes, minlength=data.num_items).float()
    graph_rows.append({
        'graph_rating': threshold, 'edges': undirected_edges,
        'mean_item_degree': degree.mean().item(),
        'max_item_degree': degree.max().item(),
        'isolated_items': int((degree == 0).sum()),
    })
pd.DataFrame(graph_rows)

## 2. 随机种子为什么重要？

种子同时控制 embedding 初始化、每轮样本顺序和 BPR 负采样。某个未观察电影可能是用户尚未见过的潜在正例；换种子会改变它是否以及何时被当成负样本。因此我们比较的是多次实验的均值与标准差，而不是挑选最好的 seed。固定 seed 主要用于复现，不会让采样偏差消失。

In [ ]:
# 默认先跑可快速检查的矩阵；确认流程后改成 True 获得正式三种子结果。
RUN_FULL = False
seeds = [42, 43, 44] if RUN_FULL else [42]
max_epochs = 250 if RUN_FULL else 80
patience = 100 if RUN_FULL else 40
configs = [
    {'name': 'L0-no-propagation', 'layers': 0, 'alpha': None},
    {'name': 'L1', 'layers': 1, 'alpha': None},
    {'name': 'L2', 'layers': 2, 'alpha': None},
    {'name': 'L3-uniform', 'layers': 3, 'alpha': None},
    {'name': 'L3-ego-heavy', 'layers': 3, 'alpha': torch.tensor([0.55, 0.25, 0.15, 0.05])},
]
len(seeds) * len(configs) * len(datasets)

## 3. 深度与层权重实验

0 层只学习用户/物品自身 embedding，接近 MF；1–3 层逐步加入邻居信息。均匀三层的四个表示各占 25%，`ego-heavy` 则保留 55% 原始 embedding，减少远端热门节点对个性表示的冲淡。epoch 只优化参数，不会改变传播跳数。

In [ ]:
rows, histories = [], {}
for graph_rating, data in datasets.items():
    for config in configs:
        for seed in seeds:
            seed_everything(seed)
            model = build_lightgcn(
                data.num_users, data.num_items, embedding_dim=64,
                num_layers=config['layers'], alpha=config['alpha'],
            )
            result = train_recommender(
                model, data, device=device, seed=seed, max_epochs=max_epochs,
                patience=patience, k=20,
            )
            model = model.to(device).eval()
            device_data = data.to(device)
            diagnostics = ranking_diagnostics(
                model.get_embedding(device_data.edge_index), device_data, split='test', k=20
            )
            rows.append({
                'graph_rating': graph_rating, 'model': config['name'], 'seed': seed,
                'best_epoch': result.best_epoch,
                'validation_ndcg@20': result.validation.ndcg,
                'test_recall@20': diagnostics.metrics.recall,
                'test_ndcg@20': diagnostics.metrics.ndcg,
                'coverage@20': diagnostics.coverage,
                'average_popularity': diagnostics.average_popularity,
                'head_recall@20': diagnostics.head_recall,
                'tail_recall@20': diagnostics.tail_recall,
            })
            histories[(graph_rating, config['name'], seed)] = result.history
results = pd.DataFrame(rows)
results.sort_values('validation_ndcg@20', ascending=False)

In [ ]:
metric_columns = [
    'validation_ndcg@20', 'test_recall@20', 'test_ndcg@20', 'coverage@20',
    'average_popularity', 'head_recall@20', 'tail_recall@20',
]
summary = results.groupby(['graph_rating', 'model'])[metric_columns].agg(['mean', 'std'])
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for (threshold, name, seed), history in histories.items():
    if seed == seeds[0]:
        frame = pd.DataFrame(history)
        axes[0].plot(frame.epoch, frame.val_ndcg, label=f'>={threshold} {name}')
first_seed = results[results.seed == seeds[0]]
for threshold, group in first_seed.groupby('graph_rating'):
    axes[1].scatter(group['coverage@20'], group['test_ndcg@20'], label=f'graph >= {threshold}')
    for _, row in group.iterrows():
        axes[1].annotate(row['model'], (row['coverage@20'], row['test_ndcg@20']), fontsize=7)
axes[0].set(xlabel='epoch', ylabel='validation NDCG@20', title='优化是否仍在继续')
axes[1].set(xlabel='catalogue coverage@20', ylabel='test NDCG@20', title='准确率—覆盖率权衡')
axes[0].legend(fontsize=7, ncol=2); axes[1].legend(); plt.tight_layout()

## 4. 如何解释结果

1. `>=3` 胜过 `>=4`：额外行为边提供了可用协同信号；反之则说明中性评分主要带来噪声。
2. L1/L2 胜过 L0：邻居传播确实有贡献；L3 回落通常意味着远端噪声或过平滑。
3. ego-heavy 胜过 uniform：传播有用，但当前图不值得让邻居主导表示。
4. head recall 高、tail recall 低且 average popularity 高：模型主要依赖热门电影。
5. NDCG 高但 coverage 低：模型可能准确，却把曝光集中在很少的商品上。电商还应把点击、加购、购买建成不同关系，并进行时间切分和在线 A/B 测试。

正式结论只看 validation 选择配置，test 用于最后一次汇报；不要根据 test 反复调阈值。